# Generator

En python les `generator` sont des `lazy iterable`, l'objet `generator` pourra être parcouru dans une boucle `for` ou avec la built-in `next`.
Le parcours est a sens et les objets ne sont pas des `sequences` il n'y a donc pas de notion d'index

## Generator expression

Comme les `list` ou les `dict` on peux écrirer des generator comprehension.

In [1]:
numbers = [ 1,2,3,4 ]

powers = ( n ** 2 for n in numbers )

powers

<generator object <genexpr> at 0x7fb0e5f48f20>

Ici on vient de créer une `generator expression`, tant qu'on ne consomme pas l'objet on aura pas les valeurs qui sont produites.

Pour consommer l'objet on peux utiliser `next`, une boucle `for` ou un constructeur d'`iterable` ( ex `list` )

In [2]:
powers = ( n ** 2 for n in numbers )
value = next( powers )
print("Next :", value)
i = 0
for value in powers:
    i+=1
    print("For :", value)
    if i > 1:
        break
print( "Tuple: ", tuple(powers) )
print( "List: ", list(powers) )

Next : 1
For : 4
For : 9
Tuple:  (16,)
List:  []


Une fois que le `generator` est complétement consommé, il enverra systématiquement une exception `StopIteration`

In [3]:
try:
    value = next(powers)
    print(value)
except StopIteration as e:
    print("exception de type StopIteration")

exception de type StopIteration


### map and filter

Les built-in `map` et `filter` produisent des `lazy iteranle` et se comporte comme les `generators`.   
La méthode `map` permet d'appliquer une fonction a un `iterable`.

In [4]:
# Exemple concret AUU
for custom in [ "mars", "rouge", "sia", "bunker" ]:
    if any( map(custom.startswith,["rouge", "violet", "bunker", "monetique", "bleu", "bpcesa"])):
        print(custom, "startswith")

rouge startswith
bunker startswith


La methode `filter` permet d'appliquer une méthode de filtre sur un `iterable`, une méthode de filtre renvoie un `boolean` pour indiquer si on conserve l'émeent.  
Un exemple simple:

In [5]:
def is_odd(n):
    return n % 2 == 0

print(tuple(filter( is_odd, range(12) ) ))

def square(n):
    return n ** 2

print( list( map( square, filter( is_odd, range(12) ))))

(0, 2, 4, 6, 8, 10)
[0, 4, 16, 36, 64, 100]


On peut aussi écrire des filtre plus complexe en utiliser une `inner fonction` qui dertemine le filtrage.  

Un exemple pour filtrer une liste de `dict` sans connaitre à l'avance son contenu.  

In [6]:
def my_custom_filter( iterable, **filters):
    def inner_filter( item ):
        # We check that all filter keys match item key
        if any( key not in item.keys() for key in filters ):
            return False
        # We check that item match all filters
        return all( item[key] in values for key,values in filters.items() )
    return filter( inner_filter, iterable )

values = [ {"a": 1,
            "b": 2,},
        {"a": 2,},
        {"a": 1,
         "b": 3, },
        {"a": 2,
         "b": 2,},]

print( list( my_custom_filter( values, a=(1,) )))

print( list( my_custom_filter( values, a=(1,), b=(3,) )))

[{'a': 1, 'b': 2}, {'a': 1, 'b': 3}]
[{'a': 1, 'b': 3}]


In [7]:
import random
import time

values = list(range(10))
print( "Large value creation Start" )
large_values = list( { "a": random.choice(values) } for _ in range(1000000))
print( f"Large value creation End {len(large_values)} items" )

start = time.perf_counter()
my_filter = my_custom_filter( large_values, a=(1,0,) )
finish = time.perf_counter()
print( f"Filter creation in {round(finish-start, 2)} second(s)" )

start = time.perf_counter()
first_item = next(my_filter)
finish = time.perf_counter()
print(f'First item : {first_item} in {round(finish-start, 2)} second(s)')

start = time.perf_counter()
print( len( list( my_filter ) ) )
finish = time.perf_counter()
print(f'all items filtered in {round(finish-start, 2)} second(s)')


Large value creation Start
Large value creation End 1000000 items
Filter creation in 0.0 second(s)
First item : {'a': 0} in 0.0 second(s)
199906
all items filtered in 1.16 second(s)


### other iterator

La buitl-in `enumerate` retourne un `iterator` les objet File sont des `iterator`.  
Il y a une différence entre `for line in file` et `for line in file.readlines()`.  

La fonction readlines retourne une `list` et donc vous va parcourir la totalité de l'objet en amont alors que si vous exploitez directement l'`iterator` de l'objet file, vous ne ferez qu'un parcours.

## generator function

Pour écrire un generator via une fonction il faut utiliser le mot clef `yield` à la place de `return`.  

Les calculs ne sont effectué qu'a chaque fois qu'on le sollicite et s'arréte a chaque `yield` et de la même maniére si le `generator` est épuisé il enverra un `StopIteration`

In [8]:
def test_gen():
    print( "Start" )
    print( "Value 1" )
    yield 1
    print( "Value 2" )
    yield 2
    print( "Value 3" )
    yield 3
    print( "End" )

for value in test_gen():
    print("Generator returned: ", value)

Start
Value 1
Generator returned:  1
Value 2
Generator returned:  2
Value 3
Generator returned:  3
End


# Exercice

Ecrire une fonction compact qui prend un iterable en paramétre et supprime ces doublons lors du parcours.

```
list(compact( [ 1,1,1,2,2,2,1,1,3 ] ) ) => [1,2,1,3]
```

In [ ]:
def compact( iterable ):
    # Your code here

In [ ]:
g_compact = compact([1,1,2,2,3,1])
assert type(g_compact) == type(iter(g_compact))
assert list(g_compact) == [1,2,3,1]